# Local Expedia analytics bootstrap

Один раз запускаем этот notebook из **корня проекта**. Он:

1. при необходимости конвертирует CSV → Parquet через DuckDB без загрузки всего файла в pandas;
2. создаёт `data/analytics.duckdb`;
3. создаёт схемы `raw`, `staging`, `marts`, `scratch`, `meta`;
4. создаёт `raw.*` views поверх Parquet.

Исходные CSV и Parquet не изменяются.

In [2]:
%pip install duckdb pyarrow

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
PARQUET = DATA / 'parquet'
DB_PATH = DATA / 'analytics.duckdb'

PARQUET.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA / 'train.csv'
TEST_CSV = DATA / 'test.csv'
DEST_CSV = DATA / 'destinations.csv'

TRAIN_PARQUET = PARQUET / 'train.parquet'
TEST_PARQUET = PARQUET / 'test.parquet'
DEST_PARQUET = PARQUET / 'destinations.parquet'

ROOT, DB_PATH

(PosixPath('/home/neukluzhiy/Desktop/Projects/HotelsBooking'),
 PosixPath('/home/neukluzhiy/Desktop/Projects/HotelsBooking/data/analytics.duckdb'))

In [4]:
def sql_path(path):
    return "'" + str(path).replace("'", "''") + "'"

def csv_to_parquet(src, dst):
    if dst.exists():
        print(f'skip: {dst.name} already exists')
        return
    if not src.exists():
        print(f'skip: {src.name} not found')
        return

    con = duckdb.connect()
    try:
        con.execute(f"""
            COPY (
                SELECT *
                FROM read_csv(
                    {sql_path(src)},
                    header = true,
                    delim = ',',
                    sample_size = 1000000,
                    strict_mode = false,
                    null_padding = true,
                    ignore_errors = true
                )
            )
            TO {sql_path(dst)}
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)
        """)
    finally:
        con.close()

    print(f'created: {dst}')

In [5]:
csv_to_parquet(TRAIN_CSV, TRAIN_PARQUET)
csv_to_parquet(TEST_CSV, TEST_PARQUET)
csv_to_parquet(DEST_CSV, DEST_PARQUET)

InvalidInputException: Invalid Input Error: Error when sniffing file "/home/neukluzhiy/Desktop/Projects/HotelsBooking/data/train.csv".
It was not possible to automatically detect the CSV parsing dialect
The search space used was:
Delimiter Candidates: ',', '|', ';', '	'
Quote/Escape Candidates: ['(no quote)','(no escape)'],['"','(no escape)'],['"','"'],['"','''],['"','\'],[''','(no escape)'],[''','''],[''','"'],[''','\']
Comment Candidates: '\0', '#'
Encoding: utf-8
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Set delimiter (e.g., delim=',')
* Set quote (e.g., quote='"')
* Set escape (e.g., escape='"')
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable ignore errors (ignore_errors=true) to ignore potential errors
* Enable null padding (null_pad

Если Parquet уже был создан раньше, предыдущая ячейка его не перезаписывает.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ['raw', 'staging', 'marts', 'scratch', 'meta']:
    con.execute(f'CREATE SCHEMA IF NOT EXISTS {schema}')

sources = {
    'train': TRAIN_PARQUET,
    'test': TEST_PARQUET,
    'destinations': DEST_PARQUET,
}

for name, path in sources.items():
    if path.exists():
        con.execute(f"""
            CREATE OR REPLACE VIEW raw.{name} AS
            SELECT * FROM read_parquet({sql_path(path)})
        """)
        print(f'raw.{name} -> {path.name}')

con.close()

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)

objects = con.sql("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema IN ('raw', 'staging', 'marts', 'scratch', 'meta')
    ORDER BY table_schema, table_name
""").df()

display(objects)
con.close()

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
display(con.sql('DESCRIBE raw.train').df())
con.close()

## Sanity check

Небольшой запрос, чтобы проверить, что агент сможет читать базу. Полный датасет в pandas не загружается.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)

check = con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user_id) AS users,
        MIN(date_time) AS min_date,
        MAX(date_time) AS max_date
    FROM raw.train
""").df()

display(check)
con.close()